## Calculate Days Since Previous Ticket

This cell creates the `user_ticket_gaps` table in the gold layer by:
* Joining support tickets with user information from the silver layer
* Using window functions (LAG) to identify the previous ticket for each user
* Calculating the number of days between consecutive tickets
* Filtering to show only records where there was a previous ticket

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databrickscourse.gold_final_dev_table;

CREATE OR REPLACE TABLE databrickscourse.gold_final_dev_table.user_ticket_gaps AS
WITH ticket_gaps AS (
  SELECT 
    st.user_id,
    u.first_name,
    u.last_name,
    u.email,
    u.country,
    u.signup_date,
    st.ticket_id,
    st.created_date,
    st.category,
    st.priority,
    st.resolved_date,
    st.resolution_time_hours,
    st.satisfaction_score,
    LAG(st.ticket_id) OVER (PARTITION BY st.user_id ORDER BY st.created_date) as previous_ticket_id,
    LAG(st.created_date) OVER (PARTITION BY st.user_id ORDER BY st.created_date) as previous_ticket_date,
    DATEDIFF(
      st.created_date, 
      LAG(st.created_date) OVER (PARTITION BY st.user_id ORDER BY st.created_date)
    ) as days_since_previous_ticket
  FROM databrickscourse.silver_cleaned_data.support_tickets_cleaned st
  INNER JOIN databrickscourse.silver_cleaned_data.users_cleaned u
    ON st.user_id = u.user_id
)
SELECT 
  user_id,
  first_name,
  last_name,
  email,
  country,
  signup_date,
  ticket_id,
  created_date,
  category,
  priority,
  resolved_date,
  resolution_time_hours,
  satisfaction_score,
  previous_ticket_id,
  previous_ticket_date,
  days_since_previous_ticket
FROM ticket_gaps
WHERE days_since_previous_ticket IS NOT NULL
ORDER BY user_id, created_date;

SELECT * FROM databrickscourse.gold_final_dev_table.user_ticket_gaps

user_id,first_name,last_name,email,country,signup_date,ticket_id,created_date,category,priority,resolved_date,resolution_time_hours,satisfaction_score,previous_ticket_id,previous_ticket_date,days_since_previous_ticket
USR_1015,Richard,Moore,richard.moore15@example.com,UK,2023-05-31,TKT_030015,2024-07-14,onboarding,low,2024-07-29,368,4,TKT_030030,2024-03-26,110
USR_1021,Edward,Rodriguez,edward.rodriguez21@example.com,AU,2024-06-10,TKT_030010,2024-08-24,billing,low,null,null,null,TKT_030023,2024-01-31,206
USR_1022,Nancy,Davis,nancy.davis22@example.com,AU,2022-10-25,TKT_030027,2024-07-12,technical,medium,2024-08-10,706,2,TKT_030003,2024-06-06,36
USR_1022,Nancy,Davis,nancy.davis22@example.com,AU,2022-10-25,TKT_030016,2024-07-17,onboarding,high,2024-07-18,27,2,TKT_030027,2024-07-12,5
USR_1022,Nancy,Davis,nancy.davis22@example.com,AU,2022-10-25,TKT_030005,2024-12-12,account,low,2024-12-19,179,4,TKT_030016,2024-07-17,148
USR_1028,Margaret,Garcia,margaret.garcia28@example.com,US,2024-08-01,TKT_030031,2024-04-17,billing,high,2024-04-19,70,1,TKT_030006,2024-02-18,59
USR_1028,Margaret,Garcia,margaret.garcia28@example.com,US,2024-08-01,TKT_030002,2024-07-27,account,high,null,null,null,TKT_030031,2024-04-17,101
USR_1030,Karen,Clark,karen.clark30@example.com,AU,2024-11-19,TKT_030037,2024-11-30,billing,critical,null,null,null,TKT_030014,2024-05-23,191
USR_1037,Margaret,Miller,margaret.miller37@example.com,CA,2023-03-02,TKT_030025,2024-06-03,billing,high,2024-06-09,162,4,TKT_030008,2024-02-08,116
USR_1037,Margaret,Miller,margaret.miller37@example.com,CA,2023-03-02,TKT_030024,2024-09-26,technical,low,2024-09-29,75,4,TKT_030025,2024-06-03,115


## Join Users and Support Tickets

This cell creates the `final_joined_table` in the gold layer by:
* Performing an inner join between users and support tickets from the silver layer
* Including all user details (personal info, signup date, referral source)
* Including all support ticket details (ticket info, resolution metrics, satisfaction scores)
* Preserving metadata columns (_row, _fivetran_synced) from both source tables
* Ordering results by user_id and ticket created_date

In [0]:
%sql
CREATE OR REPLACE TABLE databrickscourse.gold_final_dev_table.final_joined_table AS
SELECT 
  u.user_id,
  u.first_name,
  u.last_name,
  u.email,
  u.country,
  u.signup_date,
  u.referral_source,
  u._row as user_row,
  u._fivetran_synced as user_fivetran_synced,
  st.ticket_id,
  st.created_date,
  st.category,
  st.priority,
  st.resolved_date,
  st.resolution_time_hours,
  st.satisfaction_score,
  st._row as ticket_row,
  st._fivetran_synced as ticket_fivetran_synced
FROM databrickscourse.silver_cleaned_data.users_cleaned u
INNER JOIN databrickscourse.silver_cleaned_data.support_tickets_cleaned st
  ON u.user_id = st.user_id
ORDER BY u.user_id, st.created_date;

SELECT * FROM databrickscourse.gold_final_dev_table.final_joined_table

user_id,first_name,last_name,email,country,signup_date,referral_source,user_row,user_fivetran_synced,ticket_id,created_date,category,priority,resolved_date,resolution_time_hours,satisfaction_score,ticket_row,ticket_fivetran_synced
USR_1001,David,White,david.white1@example.com,AU,2022-08-12,organic,2,2026-05-02T10:07:01.275Z,TKT_030017,2024-08-29,feature_request,low,2024-09-22,599,null,18,2026-05-02T10:16:37.245Z
USR_1002,Grace,Wilson,grace.wilson2@example.com,US,2022-03-21,organic,3,2026-05-02T10:07:01.275Z,TKT_030035,2024-06-12,feature_request,low,2024-06-16,108,2,36,2026-05-02T10:16:37.251Z
USR_1004,Emma,Miller,emma.miller4@example.com,UK,2024-05-23,social_media,5,2026-05-02T10:07:01.276Z,TKT_030009,2024-03-06,technical,critical,null,null,null,10,2026-05-02T10:16:37.245Z
USR_1008,Daniel,Martin,daniel.martin8@example.com,US,2023-07-10,social_media,9,2026-05-02T10:07:01.276Z,TKT_030001,2024-03-20,onboarding,low,null,null,null,2,2026-05-02T10:16:37.244Z
USR_1009,Edward,Jones,edward.jones9@example.com,US,2024-02-29,partner,10,2026-05-02T10:07:01.276Z,TKT_030013,2024-11-27,billing,medium,2024-12-14,422,null,14,2026-05-02T10:16:37.245Z
USR_1010,Christopher,Clark,christopher.clark10@example.com,AU,2024-01-25,google_ads,11,2026-05-02T10:07:01.276Z,TKT_030018,2024-12-02,billing,medium,2024-12-24,547,1,19,2026-05-02T10:16:37.245Z
USR_1014,Daniel,Miller,daniel.miller14@example.com,US,2023-07-15,partner,15,2026-05-02T10:07:01.276Z,TKT_030022,2024-01-01,billing,critical,2024-01-27,629,2,23,2026-05-02T10:16:37.246Z
USR_1015,Richard,Moore,richard.moore15@example.com,UK,2023-05-31,social_media,16,2026-05-02T10:07:01.276Z,TKT_030030,2024-03-26,feature_request,high,2024-03-29,91,3,31,2026-05-02T10:16:37.250Z
USR_1015,Richard,Moore,richard.moore15@example.com,UK,2023-05-31,social_media,16,2026-05-02T10:07:01.276Z,TKT_030015,2024-07-14,onboarding,low,2024-07-29,368,4,16,2026-05-02T10:16:37.245Z
USR_1020,Barbara,Rodriguez,barbara.rodriguez20@example.com,UK,2022-10-27,partner,21,2026-05-02T10:07:01.276Z,TKT_030020,2024-06-29,feature_request,low,2024-07-23,578,2,21,2026-05-02T10:16:37.246Z
